# SQL-LM: Train a 247M-parameter SQL-specialist LLM from scratch

Full pipeline — pretrain → SFT → GRPO — driven from a single notebook.
Each cell is **idempotent**: safe to re-run after a Colab session disconnects.

| Stage | Script | Skip condition |
|-------|--------|----------------|
| 4a | `prepare_pretrain_data.py` | `.done` marker present locally or on Drive |
| Spider | gdown | `spider/dev.json` present locally or zip on Drive |
| 6a | `prepare_sft_data.py` | `.done` marker |
| 7a | `prepare_rl_prompts.py` | `.done` marker |
| 5 | `pretrain_base.py` | checkpoint valid + `step ≥ train_steps` |
| 6b | `train_sft.py` | checkpoint valid + all epochs complete |
| 7b | `train_grpo.py` | checkpoint valid + `step ≥ iterations` |

**GPU required for full run.** Runtime → Change runtime type → T4 GPU.

## Setup

In [ ]:
# Install extra packages (torch is pre-installed in Colab).
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "tiktoken", "datasets", "h5py", "tqdm", "wandb"],
    check=True,
)

from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
import os

REPO = "/content/llm_from_scratch_swapd"

if not os.path.isdir(REPO):
    # Replace with your actual repository URL.
    os.system(f"git clone https://github.com/YOUR_USERNAME/llm_from_scratch_swapd {REPO}")
else:
    os.system(f"git -C {REPO} pull --ff-only")
    print(f"Repo already at {REPO}")

os.chdir(REPO)
print("Working directory:", os.getcwd())

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# SMOKE = True : tiny synthetic run, ~2 min on CPU, no downloads needed.
#                Use this to verify the full pipeline end-to-end.
# SMOKE = False: full training run — GPU required, Spider download needed.

SMOKE = True   # ← TOGGLE HERE

import os, shutil, json

os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO

if SMOKE:
    DATA_DIR        = "/tmp/sql_lm_smoke"
    CKPT_DIR        = "/tmp/sql_lm_smoke"
    LOG_DIR         = "/tmp/sql_lm_smoke/logs"
    HF_CACHE        = "/tmp/hf_cache"
    DRIVE_CKPT      = ""   # no Drive sync in smoke mode
    DRIVE_DATA      = ""
    SPIDER_DIR      = ""   # not needed — smoke builds its own SQLite
    CONFIG_BASE     = "configs/smoke/base.json"
    CONFIG_PRETRAIN = "configs/smoke/pretrain.json"
    CONFIG_SFT      = "configs/smoke/sft.json"
    CONFIG_GRPO     = "configs/smoke/grpo.json"
else:
    DATA_DIR        = "/content/data"
    CKPT_DIR        = "/content/ckpts"
    LOG_DIR         = "/content/logs"
    HF_CACHE        = "/content/hf_cache"
    DRIVE_CKPT      = "/content/drive/MyDrive/sql_lm/ckpts"
    DRIVE_DATA      = "/content/drive/MyDrive/sql_lm/data"
    SPIDER_DIR      = os.path.join(DATA_DIR, "spider")
    CONFIG_BASE     = "configs/base.json"
    CONFIG_PRETRAIN = "configs/pretrain.json"
    CONFIG_SFT      = "configs/sft.json"
    CONFIG_GRPO     = "configs/grpo.json"

for d in [DATA_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
if DRIVE_CKPT:
    os.makedirs(DRIVE_CKPT, exist_ok=True)
if DRIVE_DATA:
    os.makedirs(DRIVE_DATA, exist_ok=True)

# Checkpoint paths (match the JSON configs).
PRETRAIN_CKPT = os.path.join(CKPT_DIR, "pretrain.pt")
SFT_CKPT      = os.path.join(CKPT_DIR, "sft.pt")
GRPO_CKPT     = os.path.join(CKPT_DIR, "grpo.pt")

# Read total-step counts from configs so completion detection works across
# smoke and full modes without hard-coding.
def _cfg_int(path, key, default=0):
    try:
        return json.loads(open(path).read()).get(key, default) or default
    except Exception:
        return default

PRETRAIN_TOTAL = _cfg_int(CONFIG_PRETRAIN, "train_steps") or 20000
GRPO_TOTAL     = _cfg_int(CONFIG_GRPO,     "iterations")  or 2000
SFT_EPOCHS     = _cfg_int(CONFIG_SFT,      "epochs")      or 3

print(f"Mode : {'SMOKE' if SMOKE else 'FULL'}")
print(f"Data : {DATA_DIR}")
print(f"Ckpts: {CKPT_DIR}")
print(f"Drive ckpts: {DRIVE_CKPT or '(disabled)'}")
print(f"Drive data : {DRIVE_DATA or '(disabled)'}")
print(f"Pretrain steps: {PRETRAIN_TOTAL}  SFT epochs: {SFT_EPOCHS}  GRPO iters: {GRPO_TOTAL}")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
import subprocess, sys, torch


def _run(cmd: list) -> None:
    """Run cmd with streaming output visible in the notebook cell."""
    proc = subprocess.Popen(
        cmd,
        env={**os.environ, "PYTHONPATH": REPO},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command exited {proc.returncode}: {' '.join(str(c) for c in cmd)}")


def _ckpt_is_valid(path: str, expected_stage: str = None) -> bool:
    """
    True iff path exists, torch.load succeeds, and 'stage'/'step' keys are present.
    A session crash mid-write leaves a partial file that fails to load or is
    missing these keys — both cases return False here.
    """
    if not os.path.exists(path):
        return False
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        if "stage" not in ckpt or "step" not in ckpt:
            return False
        if expected_stage and ckpt.get("stage") != expected_stage:
            return False
        return True
    except Exception:
        return False


def _training_needed(path: str, stage: str, total_steps: int = None, sft_mode: bool = False):
    """
    Returns (needs_to_run: bool, should_resume: bool).
    Also removes corrupt checkpoints in place so --resume latest won't crash.

    Completion check:
      - pretrain / grpo : step >= total_steps
      - sft             : ckpt['epoch'] >= cfg.epochs  (sft_mode=True)
    """
    if not os.path.exists(path):
        return True, False
    if not _ckpt_is_valid(path, stage):
        print(f"  [warn] Corrupt checkpoint at {path!r} — removing and restarting.")
        os.remove(path)
        return True, False
    # Checkpoint is structurally valid — check completion.
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if sft_mode:
        # save_stage_ckpt flattens extra=dict via payload.update(), so
        # 'epoch' is at the top level of the checkpoint, not nested under 'extra'.
        done_epochs = ckpt.get("epoch", 0)
        cfg_epochs  = ckpt.get("cfg",   {}).get("epochs", SFT_EPOCHS)
        if done_epochs >= cfg_epochs:
            print(f"  SFT complete ({done_epochs}/{cfg_epochs} epochs) — skipping.")
            return False, False
        print(f"  SFT epoch {done_epochs}/{cfg_epochs} — will resume.")
        return True, True
    if total_steps is not None:
        done = ckpt.get("step", 0)
        if done >= total_steps:
            print(f"  Complete (step {done}/{total_steps}) — skipping.")
            return False, False
        print(f"  Step {done}/{total_steps} — will resume.")
        return True, True
    # No completion criterion — always resume if valid.
    return True, True


def _data_is_ready(local_path: str, drive_dir: str = "") -> bool:
    """
    True iff local_path + its .done marker both exist.
    If the marker is on Drive but the file is not local, restores both first.
    The marker is written only after a complete successful write, so its
    presence implies the data file is intact — unlike a bare existence check.
    """
    marker_local = local_path + ".done"
    if os.path.exists(marker_local) and os.path.exists(local_path):
        return True
    if drive_dir:
        marker_drive = os.path.join(drive_dir, os.path.basename(marker_local))
        drive_copy   = os.path.join(drive_dir, os.path.basename(local_path))
        if os.path.exists(marker_drive) and os.path.exists(drive_copy):
            print(f"  [cache] Restoring {os.path.basename(local_path)} from Drive …")
            shutil.copy2(drive_copy,   local_path)
            shutil.copy2(marker_drive, marker_local)
            return True
    return False


def _mark_done(local_path: str, drive_dir: str = "") -> None:
    """Write .done marker then sync both file and marker to Drive."""
    marker = local_path + ".done"
    open(marker, "w").close()
    if drive_dir:
        os.makedirs(drive_dir, exist_ok=True)
        shutil.copy2(local_path, os.path.join(drive_dir, os.path.basename(local_path)))
        shutil.copy2(marker,     os.path.join(drive_dir, os.path.basename(marker)))
        print(f"  [sync] Drive ← {os.path.basename(local_path)}")


print("Helpers ready.")

## Data Pipeline

All data preparation cells are **idempotent**: they check for a `.done` marker
before running. On reconnect, cached files are restored from Drive automatically.

In [ ]:
# ── Stage 4a: Pretrain tokenised data ────────────────────────────────────────
# FineWeb-Edu (2.6 B tokens) + the-stack-dedup (15 % code) → flat int32 HDF5.
# Full run: several hours (download + tokenise). Smoke: ~2 s (synthetic).

PRETRAIN_TRAIN = os.path.join(DATA_DIR, "pretrain_train.h5")
PRETRAIN_DEV   = os.path.join(DATA_DIR, "pretrain_dev.h5")

if _data_is_ready(PRETRAIN_TRAIN, DRIVE_DATA) and _data_is_ready(PRETRAIN_DEV, DRIVE_DATA):
    print("Pretrain HDF5 already cached — skipping data prep.")
else:
    cmd = [sys.executable, "scripts/prepare_pretrain_data.py",
           "--out_dir", DATA_DIR, "--hf_cache", HF_CACHE]
    if SMOKE:
        cmd.append("--smoke")
    _run(cmd)
    _mark_done(PRETRAIN_TRAIN, DRIVE_DATA)
    _mark_done(PRETRAIN_DEV,   DRIVE_DATA)

In [ ]:
# ── Spider dataset ────────────────────────────────────────────────────────────
# Required for SFT SQL examples and GRPO RL prompts.
# Smoke mode skips this — it generates a tiny synthetic SQLite instead.
#
# If the gdown ID below stops working, download spider.zip manually from:
#   https://yale-lily.github.io/spider
# and place it at /content/data/spider.zip.

if SMOKE:
    print("Smoke mode — Spider not needed.")
else:
    SPIDER_DEV_JSON  = os.path.join(SPIDER_DIR, "dev.json")
    SPIDER_ZIP_LOCAL = os.path.join(DATA_DIR, "spider.zip")
    SPIDER_ZIP_DRIVE = os.path.join(DRIVE_DATA, "spider.zip") if DRIVE_DATA else ""

    if os.path.exists(SPIDER_DEV_JSON):
        print("Spider already extracted locally — skipping.")
    elif SPIDER_ZIP_DRIVE and os.path.exists(SPIDER_ZIP_DRIVE):
        print("Restoring spider.zip from Drive …")
        shutil.copy2(SPIDER_ZIP_DRIVE, SPIDER_ZIP_LOCAL)
        subprocess.run(["unzip", "-q", SPIDER_ZIP_LOCAL, "-d", DATA_DIR], check=True)
        print("Spider extracted.")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        # Spider 1.0 from Yale NLP (verify ID at https://yale-lily.github.io/spider)
        subprocess.run(
            ["gdown", "--id", "1TqleXec_OykOYFREKKtschzY29dUcVAQ",
             "-O", SPIDER_ZIP_LOCAL],
            check=True,
        )
        subprocess.run(["unzip", "-q", SPIDER_ZIP_LOCAL, "-d", DATA_DIR], check=True)
        if SPIDER_ZIP_DRIVE:
            print("Syncing spider.zip to Drive (one-time, ~1.3 GB) …")
            shutil.copy2(SPIDER_ZIP_LOCAL, SPIDER_ZIP_DRIVE)
            print("  [sync] Drive ← spider.zip")

    assert os.path.exists(SPIDER_DEV_JSON), (
        f"Spider dev.json not found at {SPIDER_DEV_JSON!r}. "
        "Check the extraction path — Spider may extract to a sub-directory."
    )
    print(f"Spider ready at {SPIDER_DIR}")

In [ ]:
# ── Stage 6a: SFT tokenised data ─────────────────────────────────────────────
# Alpaca + Dolly + Spider + BIRD → 2-D (n_rows, context_length) int32 HDF5.
# Smoke: tiny random tokens, no download needed.

SFT_TRAIN = os.path.join(DATA_DIR, "sft_train.h5")
SFT_DEV   = os.path.join(DATA_DIR, "sft_dev.h5")

if _data_is_ready(SFT_TRAIN, DRIVE_DATA) and _data_is_ready(SFT_DEV, DRIVE_DATA):
    print("SFT HDF5 already cached — skipping data prep.")
else:
    cmd = [sys.executable, "scripts/prepare_sft_data.py",
           "--out_dir", DATA_DIR, "--hf_cache", HF_CACHE]
    if SMOKE:
        cmd.append("--smoke")
    _run(cmd)
    _mark_done(SFT_TRAIN, DRIVE_DATA)
    _mark_done(SFT_DEV,   DRIVE_DATA)

In [ ]:
# ── Stage 7a: GRPO prompt JSONL ───────────────────────────────────────────────
# Spider SQL questions → tokenised JSONL for GRPO rollouts.
# Smoke: generates a tiny SQLite (users + orders) and fake pre-tokenised prompts.

RL_TRAIN = os.path.join(DATA_DIR, "sql_prompts_train.jsonl")
RL_DEV   = os.path.join(DATA_DIR, "sql_prompts_dev.jsonl")

if _data_is_ready(RL_TRAIN, DRIVE_DATA) and _data_is_ready(RL_DEV, DRIVE_DATA):
    print("RL prompt JSONL already cached — skipping.")
else:
    cmd = [sys.executable, "scripts/prepare_rl_prompts.py",
           "--out_dir", DATA_DIR]
    if SMOKE:
        cmd.append("--smoke")
    else:
        cmd += ["--spider_dir", SPIDER_DIR]
    _run(cmd)
    _mark_done(RL_TRAIN, DRIVE_DATA)
    _mark_done(RL_DEV,   DRIVE_DATA)

## Training

Each training cell checks whether its checkpoint is valid and complete before running:
- **Absent / corrupt** → starts from scratch (corrupt files are deleted first).
- **Partial** (step < total) → resumes with `--resume latest`.
- **Complete** → skips immediately.

To force a full re-run, delete the checkpoint file and re-run the cell.

In [ ]:
# ── Stage 5: Pretrain ─────────────────────────────────────────────────────────
# Causal LM pretraining on FineWeb-Edu + code, cosine LR, bf16 autocast.
# Full: ~20 000 steps. Smoke: 20 steps on synthetic data.

assert os.path.exists(PRETRAIN_TRAIN), (
    f"Pretrain data not found at {PRETRAIN_TRAIN!r} — run Stage 4a cell first."
)

needs_run, needs_resume = _training_needed(PRETRAIN_CKPT, "pretrain", PRETRAIN_TOTAL)
if not needs_run:
    print(f"Pretrain already complete ({PRETRAIN_TOTAL} steps) — skipping.")
else:
    cmd = [
        sys.executable, "scripts/pretrain_base.py",
        "--config", CONFIG_PRETRAIN,
    ]
    if DRIVE_CKPT:
        cmd += ["--drive_ckpt_dir", DRIVE_CKPT]
    if needs_resume:
        cmd += ["--resume", "latest"]
    _run(cmd)

In [ ]:
# ── Stage 6b: Supervised fine-tuning (SFT) ───────────────────────────────────
# Fine-tunes the pretrained backbone on SQL chat examples.
# Loss masked to assistant tokens only (prompt tokens not counted).
# Full: 3 epochs. Smoke: 10 steps.

assert _ckpt_is_valid(PRETRAIN_CKPT, "pretrain"), (
    f"Pretrain checkpoint not found or invalid at {PRETRAIN_CKPT!r} — run Stage 5 first."
)
assert os.path.exists(SFT_TRAIN), (
    f"SFT data not found at {SFT_TRAIN!r} — run Stage 6a cell first."
)

needs_run, needs_resume = _training_needed(SFT_CKPT, "sft", sft_mode=True)
if not needs_run:
    print(f"SFT already complete — skipping.")
else:
    cmd = [
        sys.executable, "scripts/train_sft.py",
        "--config", CONFIG_SFT,
    ]
    if DRIVE_CKPT:
        cmd += ["--drive_ckpt_dir", DRIVE_CKPT]
    if needs_resume:
        cmd += ["--resume", "latest"]
    _run(cmd)

In [ ]:
# ── Stage 7b: GRPO ────────────────────────────────────────────────────────────
# Group-relative policy optimisation with SQL execution verifier reward.
# 4-level reward: 1.0 correct / 0.3 executes-wrong / 0.1 tags-only / 0.0 none.
# Full: 2 000 iterations.  Smoke: 5 iterations.

assert _ckpt_is_valid(SFT_CKPT, "sft"), (
    f"SFT checkpoint not found or invalid at {SFT_CKPT!r} — run Stage 6b first."
)
assert os.path.exists(RL_TRAIN), (
    f"RL prompts not found at {RL_TRAIN!r} — run Stage 7a cell first."
)

needs_run, needs_resume = _training_needed(GRPO_CKPT, "grpo", GRPO_TOTAL)
if not needs_run:
    print(f"GRPO already complete ({GRPO_TOTAL} iterations) — skipping.")
else:
    cmd = [
        sys.executable, "scripts/train_grpo.py",
        "--config", CONFIG_GRPO,
    ]
    if DRIVE_CKPT:
        cmd += ["--drive_ckpt_dir", DRIVE_CKPT]
    if needs_resume:
        cmd += ["--resume", "latest"]
    _run(cmd)

## Evaluation

Runs the SQL execution accuracy harness on all three checkpoints and prints
a comparison table: pretrain → SFT → GRPO progression.

Metrics:
- **EX%** — execution accuracy (generated SQL result set == gold result set)
- **fmt%** — fraction of responses with a well-formed `<sql>…</sql>` block
- **valid%** — fraction where the generated SQL executed without error
- **reward** — mean 4-level reward (0 / 0.1 / 0.3 / 1.0)

In [ ]:
# ── Stage 8: Evaluation ───────────────────────────────────────────────────────
# Compares pretrain / SFT / GRPO on the Spider dev split (or smoke examples).

SMOKE_DB      = os.path.join(DATA_DIR, "smoke.sqlite")
SMOKE_PROMPTS = os.path.join(DATA_DIR, "sql_prompts_train.jsonl")

cmd = [
    sys.executable, "scripts/eval_sql.py",
    "--config", CONFIG_BASE,
    "--max_new_tokens", "256",
    "--temperature",    "0.0",   # greedy for reproducibility
]

for ckpt_path, stage in [(PRETRAIN_CKPT, "pretrain"),
                          (SFT_CKPT,      "sft"),
                          (GRPO_CKPT,     "grpo")]:
    if _ckpt_is_valid(ckpt_path, stage):
        cmd += ["--ckpt", ckpt_path]
    else:
        print(f"Skipping {stage} (no valid checkpoint at {ckpt_path!r}).")

if SMOKE:
    cmd += ["--smoke", "--smoke_db", SMOKE_DB, "--smoke_prompts", SMOKE_PROMPTS]
else:
    cmd += ["--spider_dir", SPIDER_DIR, "--verbose", "--show_n", "5"]

# Write results JSON to Drive so they persist across sessions.
if DRIVE_DATA:
    os.makedirs(DRIVE_DATA, exist_ok=True)
    cmd += ["--out_json", os.path.join(DRIVE_DATA, "eval_results.json")]

_run(cmd)

## Interactive Chat

Pick a checkpoint to chat with. The REPL maintains conversation history.
Special commands: `/clear` — reset history; `/history` — show turns so far.
Type `quit` or press Ctrl-C to exit.

In [ ]:
# ── Stage 8: Interactive chat ─────────────────────────────────────────────────
# Change CHAT_CKPT to whichever checkpoint you want to probe.

CHAT_CKPT = GRPO_CKPT   # or: PRETRAIN_CKPT / SFT_CKPT

if not _ckpt_is_valid(CHAT_CKPT):
    print(f"No valid checkpoint at {CHAT_CKPT!r} — train first.")
else:
    cmd = [
        sys.executable, "scripts/chat.py",
        "--ckpt", CHAT_CKPT,
        "--config", CONFIG_BASE,
    ]
    if SMOKE:
        # Smoke uses tiny vocab — chat output is meaningless but pipeline works.
        pass
    _run(cmd)

## Tips

**Reconnecting after a Colab disconnect**
1. Run the *Setup* cells (install, Drive mount, repo, config, helpers).
2. Re-run whichever stage cell was interrupted — the skip/resume logic handles the rest.

**Checking a checkpoint's step count**
```python
import torch
ckpt = torch.load(GRPO_CKPT, map_location='cpu', weights_only=False)
print(ckpt['stage'], ckpt['step'], ckpt.get('metrics'))
```

**Running with W&B logging** (full mode)
```python
# Add to any training cell's cmd list:
cmd += ['--use_wandb', 'true', '--wandb_project', 'sql-lm']
```

**Evaluating on a single checkpoint**
```python
_run([sys.executable, 'scripts/eval_sql.py',
      '--ckpt', GRPO_CKPT,
      '--spider_dir', SPIDER_DIR,
      '--verbose', '--show_n', '10'])
```